# Negotiating the Past - Full Results Evaluation

Visualize and evaluate the classification results from the full dataset run.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

FULL_RESULTS_FILE = "data/results_mlx/full_results.csv"

print("Loading full results...")
df = pd.read_csv(FULL_RESULTS_FILE)
print(f"Total rows: {len(df):,}")

counts = df["references_past"].value_counts()
for val, cnt in counts.items():
    print(f"  {val}: {cnt:,} ({cnt/len(df)*100:.1f}%)")

# Filter to valid results
df = df[df["references_past"].isin(["yes", "no"])].reset_index(drop=True)
yes_df = df[df["references_past"] == "yes"].reset_index(drop=True)
no_df = df[df["references_past"] == "no"].reset_index(drop=True)

print(f"\nValid results: {len(df):,}")
print(f"  Yes: {len(yes_df):,}")
print(f"  No:  {len(no_df):,}")

## 2. First 100 YES Entries

In [ ]:
print("=" * 80)
print(f"FIRST 100 YES ENTRIES (out of {len(yes_df):,})")
print("=" * 80)
for i, (_, row) in enumerate(yes_df.head(100).iterrows()):
    print(f"\n[{i+1}] PROMPT: {row['prompt'][:150]}")
    justif = row["justification"][:300] if pd.notna(row["justification"]) else "N/A"
    print(f"    JUSTIFICATION: {justif}")

## 3. First 100 NO Entries

In [ ]:
print("=" * 80)
print(f"FIRST 100 NO ENTRIES (out of {len(no_df):,})")
print("=" * 80)
for i, (_, row) in enumerate(no_df.head(100).iterrows()):
    print(f"\n[{i+1}] PROMPT: {row['prompt'][:150]}")
    justif = row["justification"][:300] if pd.notna(row["justification"]) else "N/A"
    print(f"    JUSTIFICATION: {justif}")

## 4. Cluster YES Justifications

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

yes_justifications = yes_df["justification"].dropna().tolist()
print(f"Clustering {len(yes_justifications):,} YES justifications...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

vectorizer = CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))
topic_model_yes = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    nr_topics="auto",
    min_topic_size=50,
    verbose=True
)

topics_yes, probs_yes = topic_model_yes.fit_transform(yes_justifications)

print("\n" + "=" * 80)
print("YES JUSTIFICATION CLUSTERS")
print("=" * 80)
topic_info_yes = topic_model_yes.get_topic_info()
print(f"\nNumber of topics: {len(topic_info_yes)}")
print()
for _, row in topic_info_yes.head(20).iterrows():
    print(f"Topic {row['Topic']}: {row['Count']:,} docs | {row['Name']}")
    topic_words = topic_model_yes.get_topic(row["Topic"])
    if topic_words:
        words = ", ".join([w for w, _ in topic_words[:8]])
        print(f"  Keywords: {words}")
    print()

## 5. Cluster NO Justifications

In [ ]:
import random

no_justifications = no_df["justification"].dropna().tolist()

MAX_CLUSTER_SAMPLE = 100_000
if len(no_justifications) > MAX_CLUSTER_SAMPLE:
    random.seed(42)
    no_justifications_sample = random.sample(no_justifications, MAX_CLUSTER_SAMPLE)
    print(f"Sampled {MAX_CLUSTER_SAMPLE:,} from {len(no_justifications):,} NO justifications for clustering")
else:
    no_justifications_sample = no_justifications
    print(f"Clustering {len(no_justifications_sample):,} NO justifications...")

vectorizer_no = CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))
topic_model_no = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_no,
    nr_topics="auto",
    min_topic_size=50,
    verbose=True
)

topics_no, probs_no = topic_model_no.fit_transform(no_justifications_sample)

print("\n" + "=" * 80)
print("NO JUSTIFICATION CLUSTERS")
print("=" * 80)
topic_info_no = topic_model_no.get_topic_info()
print(f"\nNumber of topics: {len(topic_info_no)}")
print()
for _, row in topic_info_no.head(20).iterrows():
    print(f"Topic {row['Topic']}: {row['Count']:,} docs | {row['Name']}")
    topic_words = topic_model_no.get_topic(row["Topic"])
    if topic_words:
        words = ", ".join([w for w, _ in topic_words[:8]])
        print(f"  Keywords: {words}")
    print()